# 05.12_Phylogenetic_analysis_Python

CAFE5 演化树与节点扩张收缩可视化。

- 当前文件：`analysis/05_genome_analysis/05.12_Phylogenetic_analysis_Python.ipynb`
- 原始来源：`Codes/05.11_R_Phylogenetic_analysis.ipynb`（旧编号仅用于溯源）。
- 运行内核：**python**。
- 导入依赖：`Bio`, `csv`, `io`, `matplotlib.pyplot`, `matplotlib.ticker`, `numpy`, `pandas`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


In [ ]:
fig_dir = '/share/home/zhangze/zz/NeuralOrigin/Figures'

### 1.基因家族演化树

In [ ]:
from Bio import Phylo
import matplotlib.pyplot as plt
import csv
import io

# ===== 1. 读取 CAFE5 结果文件 =====
def load_labels(datafile):
    labels = {}
    with open(datafile) as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            taxon_id = row["#Taxon_ID"]
            labels[taxon_id] = {
                "Increase": row.get("Increase", ""),
                "Decrease": row.get("Decrease", "")
            }
    return labels

# ===== 2. 提取树结构 =====
def parse_newick(report_file):
    ids, tree = None, None
    with open(report_file) as file:
        for line in file:
            if line.startswith("# IDs of nodes:"):
                ids = Phylo.read(io.StringIO(line.replace("# IDs of nodes:", "").strip()), "newick")
            if line.startswith("Tree:"):
                tree = Phylo.read(io.StringIO(line.replace("Tree:", "").strip()), "newick")
    return ids, tree

# ===== 3. 遍历所有 clade 节点 =====
def get_clades(node):
    clades = [node]
    for clade in node.clades:
        clades += get_clades(clade)
    return clades

# ===== 4. 合并扩张 & 收缩结果到树上 =====
def annotate_tree(tree, ids, labels):
    for clade, id_clade in zip(get_clades(tree.root), get_clades(ids.root)):
        taxon_id = id_clade.name
        if taxon_id in labels:
            inc = labels[taxon_id]["Increase"]
            dec = labels[taxon_id]["Decrease"]
            clade.info = f"+{inc} / -{dec}"
        else:
            clade.info = ""
    return tree

# ===== 5. 绘制树 =====
def draw_combined_tree(tree, output_file=None):
    def label_func(n):
        return f"{n.name} ({n.info})" if n.name else n.info

    fig = plt.figure(figsize=(8, 6), frameon=False)
    Phylo.draw(tree, axes=fig.gca(), do_show=False, label_func=label_func)
    plt.title("Gene Family Expansions (+) and Contractions (-)", fontsize=14)
    if output_file:
        fig.savefig(output_file, format="png", bbox_inches="tight", dpi=300)
    else:
        plt.show()

# ===== 6. 主流程 =====
# 输入文件路径（你需要换成自己的）
clade_results = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/Gamma_clade_results.txt"
report_file  = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/Gamma_report.cafe"

labels = load_labels(clade_results)
ids, tree = parse_newick(report_file)
tree = annotate_tree(tree, ids, labels)
draw_combined_tree(tree, "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output/CAFE5_combined_tree.png")


### Node 12 收缩/扩张散点图

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

# ===== 1. 输入文件路径 =====
cafe_output = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output"
change_file = f"{cafe_output}/Gamma_change.tab"
family_file = f"{cafe_output}/Gamma_family_results.txt"

# ===== 2. 读取数据 =====
change_df = pd.read_csv(change_file, sep="\t")
family_df = pd.read_csv(family_file, sep="\t")
family_df = family_df.rename(columns={"#FamilyID": "FamilyID"})

# ===== 3. 合并数据 =====
df = change_df.merge(family_df[["FamilyID","pvalue"]], on="FamilyID")

# ===== 4. 提取特定节点（例：12） =====
node_id = "<12>"   # 注意列名要和 change.tab 对应
node_df = df[["FamilyID", node_id, "pvalue"]].copy()
node_df = node_df[node_df[node_id] != 0]   # 去掉没变化的
node_df["Change"] = node_df[node_id].astype(int)  # 保证整数型
node_df["neglogP"] = -np.log10(node_df["pvalue"])

# ===== 5. 绘制散点图 =====
plt.figure(figsize=(6,6))
x = node_df["Change"]
y = node_df["neglogP"]

colors = ["green" if x > 0 else "red" for x in node_df["Change"]]

plt.scatter(x, y, c=colors, alpha=0.5, s=20, edgecolor="k")
plt.axhline(y=-np.log10(0.05), color="grey", linestyle="--", label="p=0.05")
plt.axvline(x=0, color="black", linestyle="--")

# ===== 强制 x 轴为整数 =====
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.xlabel("Change (Contraction - / Expansion +)")
plt.ylabel("-log10(pvalue)")
plt.title("Node 12: Gene Family Contractions/Expansions")
plt.legend(loc="center left")   # 固定图例在左下角

# ===== 保存图片 =====
# 保存为PNG，300 DPI
plt.savefig(fig_dir+"/64.Scatter.node12.png", dpi=300, bbox_inches='tight', facecolor='white')
# 保存为PDF，矢量格式
plt.savefig(fig_dir+"/64.Scatter.node12.pdf", dpi=300, bbox_inches='tight', facecolor='white')

# ===== 展示图片 =====
plt.show()

### Node 15 收缩/扩张散点图

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

# ===== 1. 输入文件路径 =====
cafe_output = "/share/home/zhangze/zz/NeuralOrigin/Data/05.GenomeAnalysis/CAFE_ExpansionsContractions/CAFE_output"
change_file = f"{cafe_output}/Gamma_change.tab"
family_file = f"{cafe_output}/Gamma_family_results.txt"

# ===== 2. 读取数据 =====
change_df = pd.read_csv(change_file, sep="\t")
family_df = pd.read_csv(family_file, sep="\t")
family_df = family_df.rename(columns={"#FamilyID": "FamilyID"})

# ===== 3. 合并数据 =====
df = change_df.merge(family_df[["FamilyID","pvalue"]], on="FamilyID")

# ===== 4. 提取特定节点（例：15） =====
node_id = "<15>"   # 注意列名要和 change.tab 对应
node_df = df[["FamilyID", node_id, "pvalue"]].copy()
node_df = node_df[node_df[node_id] != 0]   # 去掉没变化的
node_df["Change"] = node_df[node_id].astype(int)  # 保证整数型
node_df["neglogP"] = -np.log10(node_df["pvalue"])

# ===== 5. 绘制散点图 =====
plt.figure(figsize=(6,6))
x = node_df["Change"]
y = node_df["neglogP"]

colors = ["green" if x > 0 else "red" for x in node_df["Change"]]

plt.scatter(x, y, c=colors, alpha=0.5, s=20, edgecolor="k")
plt.axhline(y=-np.log10(0.05), color="grey", linestyle="--", label="p=0.05")
plt.axvline(x=0, color="black", linestyle="--")

# ===== 强制 x 轴为整数 =====
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))

plt.xlabel("Change (Contraction - / Expansion +)")
plt.ylabel("-log10(pvalue)")
plt.title("Node 15: Gene Family Contractions/Expansions")
plt.legend(loc="center left")   # 固定图例在左下角


# ===== 保存图片 =====
# 保存为PNG，300 DPI
plt.savefig(fig_dir+"/64.Scatter.node15.png", dpi=300, bbox_inches='tight', facecolor='white')
# 保存为PDF，矢量格式
plt.savefig(fig_dir+"/64.Scatter.node15.pdf", dpi=300, bbox_inches='tight', facecolor='white')

# ===== 展示图片 =====
plt.show()